# Training test

A simple training test notebook, the agent is trained on **20 episodes**

llm

In [35]:
from transformers import AutoTokenizer, AutoModelForCausalLM

import torch


class FrozenLLM:
    """
    Wrapper around a frozen causal language model for prompt optimization.

    The model is loaded in inference mode and its parameters are never
    updated. The class is responsible only for loading the model and
    generating responses.

    Parameters
    ----------
    model_name : str, default="Qwen/Qwen2.5-3B-Instruct"
        Hugging Face model identifier.

    max_new_tokens : int, default=512
        Maximum number of tokens generated for each response.

    temperature : float, default=0.0
        Sampling temperature. A value of 0 uses deterministic generation.

    device_map : str, default="auto"
        Device mapping used to load the model.
    """

    def __init__(
        self,
        model_name="Qwen/Qwen2.5-3B-Instruct",
        max_new_tokens=512,
        temperature=0.0,
        device_map="auto",
    ):
        self.model_name = model_name
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature

        # ---------------------------------------------------------
        # Tokenizer
        # ---------------------------------------------------------

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name
        )
        self.tokenizer.padding_side = "left"
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token


        # Qwen may not have a pad token explicitly defined.
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = (
                self.tokenizer.eos_token
            )

        # ---------------------------------------------------------
        # Model
        # ---------------------------------------------------------

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map=device_map,
        )

        # ---------------------------------------------------------
        # Freeze model
        # ---------------------------------------------------------

        self.model.eval()

        for parameter in self.model.parameters():
            parameter.requires_grad = False

    # =============================================================
    # GENERATION CONFIGURATION
    # =============================================================

    def _generation_kwargs(self):
        """
        Build generation parameters.

        Returns
        -------
        dict
            Parameters passed to ``model.generate``.
        """

        generation_kwargs = {
            "max_new_tokens": self.max_new_tokens,
            "pad_token_id": self.tokenizer.pad_token_id,
        }

        if self.temperature > 0:
            generation_kwargs.update({
                "do_sample": True,
                "temperature": self.temperature,
            })
        else:
            generation_kwargs.update({
                "do_sample": False,
            })

        return generation_kwargs

    # =============================================================
    # SINGLE GENERATION
    # =============================================================

    def generate(self, prompt):
        """
        Generate a response for a single prompt.

        Parameters
        ----------
        prompt : str
            Input prompt.

        Returns
        -------
        str
            Generated response.
        """

        messages = [
            {
                "role": "user",
                "content": prompt,
            }
        ]

        inputs = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )

        inputs = {
            key: value.to(self.model.device)
            for key, value in inputs.items()
        }

        with torch.inference_mode():

            outputs = self.model.generate(
                **inputs,
                **self._generation_kwargs(),
            )

        input_length = inputs["input_ids"].shape[-1]

        generated_tokens = outputs[0][
            input_length:
        ]

        response = self.tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True,
        )

        return response.strip()

    # =============================================================
    # BATCH GENERATION
    # =============================================================

    def generate_batch(self, prompts):
        """
        Generate responses for multiple prompts simultaneously.

        Parameters
        ----------
        prompts : list[str]
            List of input prompts.

        Returns
        -------
        list[str]
            Generated responses in the same order as the input prompts.
        """

        if not prompts:
            return []

        messages = [
            [
                {
                    "role": "user",
                    "content": prompt,
                }
            ]
            for prompt in prompts
        ]

        # ---------------------------------------------------------
        # Tokenization
        # ---------------------------------------------------------

        inputs = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            padding=True,
        )

        inputs = {
            key: value.to(self.model.device)
            for key, value in inputs.items()
        }

        # Length of the padded input sequence.
        input_length = inputs["input_ids"].shape[1]

        # ---------------------------------------------------------
        # Generation
        # ---------------------------------------------------------

        with torch.inference_mode():

            outputs = self.model.generate(
                **inputs,
                **self._generation_kwargs(),
            )

        # ---------------------------------------------------------
        # Extract generated tokens
        # ---------------------------------------------------------

        responses = []

        for output in outputs:

            generated_tokens = output[input_length:]

            response = self.tokenizer.decode(
                generated_tokens,
                skip_special_tokens=True,
            )

            responses.append(
                response.strip()
            )

        return responses

evaluator

In [36]:
import re


class GSM8KEvaluator:
    """
    Evaluate a frozen LLM on GSM8K problems.

    The evaluator supports:

    - prompt-level caching;
    - batched LLM generation;
    - accuracy computation;
    - evaluation statistics.

    Parameters
    ----------
    llm : object
        Frozen language model exposing ``generate(prompt)``
        and ``generate_batch(prompts)`` methods.

    dataset : iterable
        GSM8K dataset containing ``question`` and ``answer`` fields.

    batch_size : int, default=8
        Number of GSM8K problems evaluated simultaneously.
    """

    def __init__(
        self,
        llm,
        dataset,
        batch_size=8,
    ):
        self.llm = llm
        self.dataset = dataset
        self.batch_size = batch_size

        # ---------------------------------------------------------
        # Prompt-level cache
        # ---------------------------------------------------------

        self.cache = {}

        # ---------------------------------------------------------
        # Evaluation statistics
        # ---------------------------------------------------------

        self.cache_hits = 0
        self.cache_misses = 0
        self.llm_calls = 0

    # =============================================================
    # EVALUATION
    # =============================================================

    def evaluate(
        self,
        prompt,
        return_details=False,
    ):
        """
        Evaluate a prompt on the GSM8K dataset.

        Parameters
        ----------
        prompt : str
            Prompt instruction to evaluate.

        return_details : bool, default=False
            If True, return detailed results for each problem.

        Returns
        -------
        float or dict
            Accuracy if ``return_details=False``.

            Otherwise, a dictionary containing accuracy,
            individual results and evaluation statistics.
        """

        # ---------------------------------------------------------
        # Prompt-level cache
        # ---------------------------------------------------------

        if prompt in self.cache:

            self.cache_hits += 1

            cached_result = self.cache[prompt]

            if return_details:
                return cached_result

            return cached_result["accuracy"]

        # ---------------------------------------------------------
        # Cache miss
        # ---------------------------------------------------------

        self.cache_misses += 1

        # ---------------------------------------------------------
        # Build all GSM8K prompts
        # ---------------------------------------------------------

        examples = list(self.dataset)

        prompts = [
            self.build_prompt(
                prompt,
                example["question"],
            )
            for example in examples
        ]

        # ---------------------------------------------------------
        # Batched generation
        # ---------------------------------------------------------

        responses = []

        for start in range(
            0,
            len(prompts),
            self.batch_size,
        ):

            batch_prompts = prompts[
                start:start + self.batch_size
            ]

            batch_responses = (
                self.llm.generate_batch(
                    batch_prompts
                )
            )

            responses.extend(
                batch_responses
            )

            self.llm_calls += len(
                batch_prompts
            )

        # ---------------------------------------------------------
        # Evaluate predictions
        # ---------------------------------------------------------

        results = []

        for example, response in zip(
            examples,
            responses,
        ):

            question = example["question"]

            target = self.extract_target_answer(
                example["answer"]
            )

            prediction = self.extract_prediction(
                response
            )

            correct = self.is_correct(
                prediction,
                target,
            )

            results.append({
                "question": question,
                "target": target,
                "response": response,
                "prediction": prediction,
                "correct": correct,
            })

        # ---------------------------------------------------------
        # Accuracy
        # ---------------------------------------------------------

        accuracy = (
            sum(
                result["correct"]
                for result in results
            )
            / len(results)
        )

        # ---------------------------------------------------------
        # Cache complete result
        # ---------------------------------------------------------

        evaluation_result = {
            "accuracy": accuracy,
            "results": results,
        }

        self.cache[prompt] = evaluation_result

        if return_details:
            return evaluation_result

        return accuracy

    # =============================================================
    # STATISTICS
    # =============================================================

    def get_statistics(self):
        """
        Return evaluator statistics.

        Returns
        -------
        dict
            Evaluation and cache statistics.
        """

        total_evaluations = (
            self.cache_hits
            + self.cache_misses
        )

        return {
            "total_evaluations": total_evaluations,
            "cache_hits": self.cache_hits,
            "cache_misses": self.cache_misses,
            "llm_calls": self.llm_calls,
            "cache_hit_rate": (
                self.cache_hits
                / total_evaluations
                if total_evaluations > 0
                else 0.0
            ),
        }

    # =============================================================
    # PROMPT BUILDING
    # =============================================================

    @staticmethod
    def build_prompt(
        prompt,
        question,
    ):
        """
        Combine the current instruction with a GSM8K question.

        Parameters
        ----------
        prompt : str
            Current optimized instruction.

        question : str
            GSM8K problem.

        Returns
        -------
        str
            Complete prompt sent to the LLM.
        """

        return f"""{prompt}

Problem:

{question}
"""

    # =============================================================
    # TARGET EXTRACTION
    # =============================================================

    @staticmethod
    def extract_target_answer(answer):
        """
        Extract the final numerical answer from a GSM8K target.

        Parameters
        ----------
        answer : str
            Original GSM8K answer containing reasoning and
            final answer.

        Returns
        -------
        str
            Normalized final answer.
        """

        match = re.search(
            r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)",
            answer,
        )

        if match is None:

            raise ValueError(
                f"Could not extract GSM8K target answer: "
                f"{answer}"
            )

        return GSM8KEvaluator.normalize_number(
            match.group(1)
        )

    # =============================================================
    # PREDICTION EXTRACTION
    # =============================================================

    @staticmethod
    def extract_prediction(response):
        """
        Extract a numerical prediction from an LLM response.

        Parameters
        ----------
        response : str
            Generated model response.

        Returns
        -------
        str or None
            Extracted numerical answer.
        """

        patterns = [
            r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)",

            r"(?:final answer|answer)"
            r"\s*(?:is|:)?\s*"
            r"([-+]?\d[\d,]*(?:\.\d+)?)",
        ]

        for pattern in patterns:

            matches = re.findall(
                pattern,
                response,
                flags=re.IGNORECASE,
            )

            if matches:

                return GSM8KEvaluator.normalize_number(
                    matches[-1]
                )

        # ---------------------------------------------------------
        # Fallback: last number
        # ---------------------------------------------------------

        numbers = re.findall(
            r"[-+]?\d[\d,]*(?:\.\d+)?",
            response,
        )

        if not numbers:
            return None

        return GSM8KEvaluator.normalize_number(
            numbers[-1]
        )

    # =============================================================
    # NUMBER NORMALIZATION
    # =============================================================

    @staticmethod
    def normalize_number(value):
        """
        Normalize a numerical answer.

        Parameters
        ----------
        value : str
            Numerical value.

        Returns
        -------
        str
            Normalized numerical representation.
        """

        value = value.strip()
        value = value.replace(",", "")

        try:

            number = float(value)

            if number.is_integer():
                return str(int(number))

            return str(number)

        except ValueError:

            return value

    # =============================================================
    # CORRECTNESS
    # =============================================================

    @staticmethod
    def is_correct(
        prediction,
        target,
    ):
        """
        Compare a prediction with the GSM8K target.

        Parameters
        ----------
        prediction : str or None
            Extracted model prediction.

        target : str
            Expected answer.

        Returns
        -------
        bool
            Whether the prediction is correct.
        """

        if prediction is None:
            return False

        return prediction == target

prompts

In [37]:
BASE_PROMPT = """Solve the following math problem.
Provide the final answer clearly."""

PROMPT_TRANSFORMATIONS = {

    0: {
        "name": "step_by_step",
        "instruction":"Solve the problem step by step."},
    1: {
        "name":"reasoning",
        "instruction":"Explain your reasoning clearly."},
    2: {
        "name":"verification",
        "instruction":"Verify your answer before giving the final answer."},
    3: {
        "name":"calculation_check",
        "instruction":"Check your calculations carefully."},
    4: {
        "name":"decomposition",
        "instruction":"Break the problem into smaller steps."},
    5: {
        "name":"relevant_information",
        "instruction":"Identify the relevant information before solving the problem."},
    6: {
        "name":"double_check",
        "instruction":"Double-check your final answer."},
    7: {
        "name":"answer_format",
        "instruction":"Clearly state the final answer at the end."
    }

}


def apply_transformation(prompt, action):
    """
    Apply a prompt transformation.
    args
    -------
    prompt : str
        current prompt.
    action : int
        ID of the transformation to apply
    Returns
    -------
    str
        Transformed prompt
    """
    transformation = PROMPT_TRANSFORMATIONS[action]["instruction"]

    if not transformation:
        return prompt
    return f"{prompt}\n\n{transformation}"

environment

In [38]:
import gymnasium as gym
import numpy as np


class PromptOptimizationEnv(gym.Env):
    """
    Gymnasium environment for prompt optimization.

    The agent sequentially selects prompt transformations.
    The LLM remains frozen and is evaluated after each transformation.

    Parameters
    ----------
    evaluator : GSM8KEvaluator
        Evaluator used to measure prompt performance.

    base_prompt : str
        Initial prompt before any transformation.

    max_steps : int, default=5
        Maximum number of transformations per episode.

    final_reward_coef : float, default=0.5
        Weight applied to the final improvement relative to
        the base prompt.
    """

    def __init__(
        self,
        evaluator,
        base_prompt,
        max_steps=5,
        final_reward_coef=0.5,
    ):
        super().__init__()

        self.evaluator = evaluator
        self.base_prompt = base_prompt
        self.max_steps = max_steps
        self.final_reward_coef = final_reward_coef

        # Number of available transformations
        self.action_dim = 8

        # ---------------------------------------------------------
        # Action space
        # ---------------------------------------------------------

        self.action_space = gym.spaces.Discrete(
            self.action_dim
        )

        # ---------------------------------------------------------
        # Observation space
        # ---------------------------------------------------------

        # Current accuracy
        # Previous accuracy
        # Step progress
        # Number of transformations already selected
        #
        # Plus one binary feature per action indicating whether
        # the transformation has already been selected.
        #
        # Total:
        # 4 + 8 = 12
        # ---------------------------------------------------------

        self.observation_dim = 12

        self.observation_space = gym.spaces.Box(
            low=0.0,
            high=1.0,
            shape=(self.observation_dim,),
            dtype=np.float32,
        )

        # ---------------------------------------------------------
        # Episode state
        # ---------------------------------------------------------

        self.current_prompt = None
        self.current_accuracy = 0.0
        self.base_accuracy = 0.0
        self.previous_accuracy = 0.0

        self.step_count = 0

        self.selected_actions = []

        self.used_actions = set()

    # =============================================================
    # OBSERVATION
    # =============================================================

    def _get_observation(self):
        """
        Build the current environment observation.

        Returns
        -------
        np.ndarray
            Current environment state.
        """

        step_progress = (
            self.step_count / self.max_steps
        )

        num_selected = (
            len(self.selected_actions)
            / self.action_dim
        )

        used_actions = np.zeros(
            self.action_dim,
            dtype=np.float32,
        )

        for action in self.used_actions:
            used_actions[action] = 1.0

        observation = np.concatenate(
            [
                np.array(
                    [
                        self.current_accuracy,
                        self.previous_accuracy,
                        step_progress,
                        num_selected,
                    ],
                    dtype=np.float32,
                ),
                used_actions,
            ]
        )

        return observation.astype(
            np.float32
        )

    # =============================================================
    # ACTION MASK
    # =============================================================

    def get_action_mask(self):
        """
        Return the mask of currently available actions.

        Returns
        -------
        np.ndarray
            Boolean mask where True means that the action
            can still be selected.
        """

        mask = np.ones(
            self.action_dim,
            dtype=bool,
        )

        for action in self.used_actions:
            mask[action] = False

        return mask

    # =============================================================
    # RESET
    # =============================================================

    def reset(
        self,
        *,
        seed=None,
        options=None,
    ):
        """
        Reset the environment.

        Returns
        -------
        observation : np.ndarray
            Initial observation.

        info : dict
            Initial environment information.
        """

        super().reset(seed=seed)

        self.current_prompt = (
            self.base_prompt
        )

        # ---------------------------------------------------------
        # Evaluate base prompt
        # ---------------------------------------------------------

        self.base_accuracy = (
            self.evaluator.evaluate(
                self.base_prompt
            )
        )

        self.current_accuracy = (
            self.base_accuracy
        )

        self.previous_accuracy = (
            self.base_accuracy
        )

        self.step_count = 0

        self.selected_actions = []

        self.used_actions = set()

        observation = (
            self._get_observation()
        )

        info = {
            "prompt": self.current_prompt,
            "accuracy": self.current_accuracy,
            "base_accuracy": self.base_accuracy,
            "actions": self.selected_actions,
        }

        return observation, info

    # =============================================================
    # STEP
    # =============================================================

    def step(self, action):
        """
        Apply a prompt transformation.

        Parameters
        ----------
        action : int
            Transformation ID.

        Returns
        -------
        observation : np.ndarray
            New environment state.

        reward : float
            Reward obtained after the transformation.

        terminated : bool
            Whether the episode naturally ended.

        truncated : bool
            Whether the episode was truncated.

        info : dict
            Environment information.
        """

        # ---------------------------------------------------------
        # Validate action
        # ---------------------------------------------------------

        if action in self.used_actions:
            raise ValueError(
                f"Action {action} has already been selected."
            )

        if not self.action_space.contains(action):
            raise ValueError(
                f"Invalid action: {action}"
            )

        # ---------------------------------------------------------
        # Previous state
        # ---------------------------------------------------------

        self.previous_accuracy = (
            self.current_accuracy
        )

        # ---------------------------------------------------------
        # Apply transformation
        # ---------------------------------------------------------

        self.current_prompt = (
            apply_transformation(
                self.current_prompt,
                action,
            )
        )

        self.used_actions.add(action)

        self.selected_actions.append(
            action
        )

        self.step_count += 1

        # ---------------------------------------------------------
        # Evaluate transformed prompt
        # ---------------------------------------------------------

        self.current_accuracy = (
            self.evaluator.evaluate(
                self.current_prompt
            )
        )

        # ---------------------------------------------------------
        # Local reward
        # ---------------------------------------------------------

        reward = (
            self.current_accuracy
            - self.previous_accuracy
        )

        # ---------------------------------------------------------
        # Episode termination
        # ---------------------------------------------------------

        terminated = (
            self.step_count
            >= self.max_steps
        )

        truncated = False

        # ---------------------------------------------------------
        # Final reward
        # ---------------------------------------------------------

        final_reward = 0.0

        if terminated:

            final_improvement = (
                self.current_accuracy
                - self.base_accuracy
            )

            final_reward = (
                self.final_reward_coef
                * final_improvement
            )

            reward += final_reward

        # ---------------------------------------------------------
        # Observation
        # ---------------------------------------------------------

        observation = (
            self._get_observation()
        )

        # ---------------------------------------------------------
        # Information
        # ---------------------------------------------------------

        info = {
            "prompt": self.current_prompt,
            "accuracy": self.current_accuracy,
            "base_accuracy": self.base_accuracy,
            "improvement": (
                self.current_accuracy
                - self.base_accuracy
            ),
            "previous_accuracy": (
                self.previous_accuracy
            ),
            "local_reward": (
                self.current_accuracy
                - self.previous_accuracy
            ),
            "final_reward": final_reward,
            "actions": list(
                self.selected_actions
            ),
        }

        return (
            observation,
            reward,
            terminated,
            truncated,
            info,
        )

policy

In [39]:
import torch
import torch.nn as nn
import numpy as np

from torch.distributions import Categorical


class PolicyNetwork(nn.Module):
    """
    Actor-Critic neural network used by the RL agent.

    The network receives the current environment state and produces:

    - action logits for the policy;
    - a state-value estimate for the critic.

    The network also supports action masking, allowing the environment
    to indicate which actions are currently available.

    Parameters
    ----------
    observation_dim : int
        Dimension of the environment observation.

    action_dim : int
        Number of available actions.

    hidden_dim : int, default=128
        Number of neurons in the hidden layers.
    """

    def __init__(
        self,
        observation_dim,
        action_dim,
        hidden_dim=128,
    ):
        super().__init__()

        # ---------------------------------------------------------
        # Shared representation
        # ---------------------------------------------------------

        self.shared_network = nn.Sequential(
            nn.Linear(
                observation_dim,
                hidden_dim,
            ),
            nn.ReLU(),

            nn.Linear(
                hidden_dim,
                hidden_dim,
            ),
            nn.ReLU(),
        )

        # ---------------------------------------------------------
        # Actor
        # ---------------------------------------------------------

        self.policy_head = nn.Linear(
            hidden_dim,
            action_dim,
        )

        # ---------------------------------------------------------
        # Critic
        # ---------------------------------------------------------

        self.value_head = nn.Linear(
            hidden_dim,
            1,
        )

    # =============================================================
    # FORWARD
    # =============================================================

    def forward(self, state):
        """
        Compute policy logits and state value.

        Parameters
        ----------
        state : torch.Tensor
            Current environment state.

        Returns
        -------
        logits : torch.Tensor
            Unnormalized action scores.

        value : torch.Tensor
            Estimated state value.
        """

        features = self.shared_network(state)

        logits = self.policy_head(features)

        value = self.value_head(features)

        return logits, value

    # =============================================================
    # ACTION MASK
    # =============================================================

    @staticmethod
    def apply_action_mask(logits, action_mask):
        """
        Mask unavailable actions.

        Parameters
        ----------
        logits : torch.Tensor
            Action logits produced by the policy.

        action_mask : torch.Tensor
            Boolean tensor where:

            True  = action available
            False = action unavailable

        Returns
        -------
        torch.Tensor
            Masked action logits.
        """

        if action_mask is None:
            return logits

        # Convert mask to boolean if necessary.
        action_mask = action_mask.bool()

        # Invalid actions receive a very negative logit.
        masked_logits = logits.masked_fill(
            ~action_mask,
            torch.finfo(logits.dtype).min,
        )

        return masked_logits

    # =============================================================
    # DISTRIBUTION
    # =============================================================

    def get_distribution(
        self,
        state,
        action_mask=None,
    ):
        """
        Build the action probability distribution.

        Parameters
        ----------
        state : torch.Tensor
            Current environment state.

        action_mask : torch.Tensor or None
            Boolean mask indicating available actions.

        Returns
        -------
        Categorical
            Probability distribution over available actions.
        """

        logits, _ = self.forward(state)

        logits = self.apply_action_mask(
            logits,
            action_mask,
        )

        return Categorical(
            logits=logits
        )

    # =============================================================
    # ACTION SELECTION
    # =============================================================

    def get_action(
        self,
        state,
        action_mask=None,
        deterministic=False,
    ):
        """
        Select an action according to the current policy.

        Parameters
        ----------
        state : torch.Tensor
            Current environment state.

        action_mask : torch.Tensor or None
            Boolean mask indicating available actions.

        deterministic : bool, default=False
            If True, select the action with the highest probability.

        Returns
        -------
        action : torch.Tensor
            Selected action.

        log_probability : torch.Tensor
            Log probability of the selected action.

        value : torch.Tensor
            Estimated state value.
        """

        logits, value = self.forward(state)

        logits = self.apply_action_mask(
            logits,
            action_mask,
        )

        distribution = Categorical(
            logits=logits
        )

        if deterministic:
            action = torch.argmax(
                logits,
                dim=-1,
            )
        else:
            action = distribution.sample()

        log_probability = distribution.log_prob(
            action
        )

        return (
            action,
            log_probability,
            value.squeeze(-1),
        )

    # =============================================================
    # EVALUATE ACTIONS
    # =============================================================

    def evaluate_actions(
        self,
        state,
        actions,
        action_mask=None,
    ):
        """
        Evaluate actions under the current policy.

        This method is used during PPO optimization.

        Parameters
        ----------
        state : torch.Tensor
            Environment states.

        actions : torch.Tensor
            Actions previously selected.

        action_mask : torch.Tensor or None
            Boolean mask indicating available actions for each state.

        Returns
        -------
        log_probabilities : torch.Tensor
            Log probabilities of the selected actions.

        entropy : torch.Tensor
            Policy entropy.

        values : torch.Tensor
            Estimated state values.
        """

        logits, values = self.forward(state)

        logits = self.apply_action_mask(
            logits,
            action_mask,
        )

        distribution = Categorical(
            logits=logits
        )

        log_probabilities = (
            distribution.log_prob(actions)
        )

        entropy = distribution.entropy()

        return (
            log_probabilities,
            entropy,
            values.squeeze(-1),
        )

    def action_mask(self):
        """
        Return a boolean mask indicating which actions are available.

        Returns:
        --------
        np.array
            Boolean action mask.
        """

        mask = np.zeros(
            self.action_space.n,
            dtype=bool
        )

        for action in self.get_available_actions():
            mask[action] = True
        return mask

ppo

In [50]:
import sys
sys.path.append('/content/drive/MyDrive/')

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm
from agent.policy import PolicyNetwork


class PPO:
    """
    Proximal Policy Optimization agent for prompt optimization.

    The agent learns to select prompt transformations that improve
    the performance of a frozen LLM on GSM8K.

    Parameters
    ----------
    env : gymnasium.Env
        Prompt optimization environment.

    learning_rate : float, default=3e-4
        Learning rate of the optimizer.

    gamma : float, default=0.99
        Discount factor.

    gae_lambda : float, default=0.95
        GAE parameter.

    clip_epsilon : float, default=0.2
        PPO clipping parameter.

    value_coef : float, default=0.5
        Weight of the value loss.

    entropy_coef : float, default=0.01
        Weight of the entropy bonus.

    update_epochs : int, default=4
        Number of optimization epochs per trajectory.

    hidden_dim : int, default=128
        Number of neurons in the policy hidden layers.

    device : str or None, default=None
        Device used for training.
    """

    def __init__(
        self,
        env,
        learning_rate=3e-4,
        gamma=0.99,
        gae_lambda=0.95,
        clip_epsilon=0.2,
        value_coef=0.5,
        entropy_coef=0.01,
        update_epochs=4,
        hidden_dim=128,
        device=None,
    ):
        self.env = env

        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.clip_epsilon = clip_epsilon

        self.value_coef = value_coef
        self.entropy_coef = entropy_coef

        self.update_epochs = update_epochs


        if device is None:
            device = (
                "cuda"
                if torch.cuda.is_available()
                else "cpu"
            )

        self.device = torch.device(device)

        # ---------------------------------------------------------
        # Environment dimensions
        # ---------------------------------------------------------

        observation_dim = (
            env.observation_space.shape[0]
        )

        action_dim = (
            env.action_space.n
        )

        # ---------------------------------------------------------
        # Policy
        # ---------------------------------------------------------

        self.policy = PolicyNetwork(
            observation_dim=observation_dim,
            action_dim=action_dim,
            hidden_dim=hidden_dim,
        ).to(self.device)

        self.optimizer = optim.Adam(
            self.policy.parameters(),
            lr=learning_rate,
        )

    # =============================================================
    # ACTION MASK
    # =============================================================

    def _get_action_mask(self):
        """
        Get the current action mask from the environment.

        Returns
        -------
        torch.Tensor
            Boolean action mask.
        """

        mask = self.env.get_action_mask()

        return torch.tensor(
            mask,
            dtype=torch.bool,
            device=self.device,
        ).unsqueeze(0)

    # =============================================================
    # ACTION SELECTION
    # =============================================================

    def select_action(
        self,
        state,
        action_mask=None,
        deterministic=False,
    ):
        """
        Select an action using the current policy.

        Parameters
        ----------
        state : np.ndarray
            Current environment state.

        action_mask : np.ndarray or None
            Boolean mask of available actions.

        deterministic : bool, default=False
            Whether to select the most probable valid action.

        Returns
        -------
        action : int
            Selected action.

        log_probability : float
            Log probability of the selected action.

        value : float
            Estimated state value.
        """

        state_tensor = torch.tensor(
            state,
            dtype=torch.float32,
            device=self.device,
        ).unsqueeze(0)

        if action_mask is not None:

            action_mask = torch.tensor(
                action_mask,
                dtype=torch.bool,
                device=self.device,
            ).unsqueeze(0)

        with torch.no_grad():

            (
                action,
                log_probability,
                value,
            ) = self.policy.get_action(
                state_tensor,
                action_mask=action_mask,
                deterministic=deterministic,
            )

        return (
            action.item(),
            log_probability.item(),
            value.item(),
        )

    # =============================================================
    # TRAJECTORY COLLECTION
    # =============================================================

    def collect_episode(self):
        """
        Collect one complete trajectory.

        Returns
        -------
        trajectory : dict
            States, actions, rewards, log probabilities,
            values, masks and terminal flags.
        """

        state, info = self.env.reset()

        trajectory = {
            "states": [],
            "actions": [],
            "rewards": [],
            "log_probs": [],
            "values": [],
            "dones": [],
            "action_masks": [],
        }

        done = False

        while not done:

            # -----------------------------------------------------
            # Get available actions
            # -----------------------------------------------------

            action_mask = self.env.get_action_mask()

            # -----------------------------------------------------
            # Select action
            # -----------------------------------------------------

            (
                action,
                log_prob,
                value,
            ) = self.select_action(
                state,
                action_mask=action_mask,
            )

            # -----------------------------------------------------
            # Environment transition
            # -----------------------------------------------------

            (
                next_state,
                reward,
                terminated,
                truncated,
                info,
            ) = self.env.step(action)

            done = (
                terminated
                or truncated
            )

            # -----------------------------------------------------
            # Store transition
            # -----------------------------------------------------

            trajectory["states"].append(state)

            trajectory["actions"].append(action)

            trajectory["rewards"].append(reward)

            trajectory["log_probs"].append(log_prob)

            trajectory["values"].append(value)

            trajectory["dones"].append(done)

            trajectory["action_masks"].append(
                action_mask
            )

            state = next_state

        return trajectory

    # =============================================================
    # ADVANTAGE ESTIMATION
    # =============================================================

    def compute_gae(self, trajectory):
        """
        Compute Generalized Advantage Estimation.

        Parameters
        ----------
        trajectory : dict
            Collected trajectory.

        Returns
        -------
        advantages : np.ndarray
            Estimated advantages.

        returns : np.ndarray
            Estimated returns.
        """

        rewards = np.asarray(
            trajectory["rewards"],
            dtype=np.float32,
        )

        values = np.asarray(
            trajectory["values"],
            dtype=np.float32,
        )

        dones = np.asarray(
            trajectory["dones"],
            dtype=np.float32,
        )

        advantages = np.zeros_like(
            rewards
        )

        gae = 0.0

        for t in reversed(
            range(len(rewards))
        ):

            if t == len(rewards) - 1:
                next_value = 0.0
            else:
                next_value = values[t + 1]

            non_terminal = (
                1.0 - dones[t]
            )

            delta = (
                rewards[t]
                + self.gamma
                * next_value
                * non_terminal
                - values[t]
            )

            gae = (
                delta
                + self.gamma
                * self.gae_lambda
                * non_terminal
                * gae
            )

            advantages[t] = gae

        returns = (
            advantages + values
        )

        return advantages, returns

    # =============================================================
    # PPO UPDATE
    # =============================================================

    def update(self, trajectory):
        """
        Update the policy using PPO.

        Parameters
        ----------
        trajectory : dict
            Collected trajectory.

        Returns
        -------
        dict
            Training statistics.
        """

        advantages, returns = (
            self.compute_gae(
                trajectory
            )
        )

        # ---------------------------------------------------------
        # Convert trajectory to tensors
        # ---------------------------------------------------------

        states = torch.tensor(
            np.asarray(
                trajectory["states"]
            ),
            dtype=torch.float32,
            device=self.device,
        )

        actions = torch.tensor(
            trajectory["actions"],
            dtype=torch.long,
            device=self.device,
        )

        old_log_probs = torch.tensor(
            trajectory["log_probs"],
            dtype=torch.float32,
            device=self.device,
        )

        action_masks = torch.tensor(
            np.asarray(
                trajectory["action_masks"]
            ),
            dtype=torch.bool,
            device=self.device,
        )

        advantages = torch.tensor(
            advantages,
            dtype=torch.float32,
            device=self.device,
        )

        returns = torch.tensor(
            returns,
            dtype=torch.float32,
            device=self.device,
        )

        # ---------------------------------------------------------
        # Advantage normalization
        # ---------------------------------------------------------

        if len(advantages) > 1:

            advantages = (
                advantages
                - advantages.mean()
            ) / (
                advantages.std()
                + 1e-8
            )

        # ---------------------------------------------------------
        # Statistics
        # ---------------------------------------------------------

        total_policy_loss = 0.0
        total_value_loss = 0.0
        total_entropy = 0.0

        # ---------------------------------------------------------
        # PPO optimization epochs
        # ---------------------------------------------------------

        for _ in range(
            self.update_epochs
        ):

            (
                new_log_probs,
                entropy,
                values,
            ) = self.policy.evaluate_actions(
                states,
                actions,
                action_mask=action_masks,
            )

            # -----------------------------------------------------
            # Probability ratio
            # -----------------------------------------------------

            ratios = torch.exp(
                new_log_probs
                - old_log_probs
            )

            # -----------------------------------------------------
            # Surrogate objectives
            # -----------------------------------------------------

            unclipped_objective = (
                ratios * advantages
            )

            clipped_ratios = torch.clamp(
                ratios,
                1.0 - self.clip_epsilon,
                1.0 + self.clip_epsilon,
            )

            clipped_objective = (
                clipped_ratios
                * advantages
            )

            policy_loss = -torch.min(
                unclipped_objective,
                clipped_objective,
            ).mean()

            # -----------------------------------------------------
            # Value loss
            # -----------------------------------------------------

            value_loss = (
                nn.functional.mse_loss(
                    values,
                    returns,
                )
            )

            # -----------------------------------------------------
            # Entropy
            # -----------------------------------------------------

            entropy_loss = (
                entropy.mean()
            )

            # -----------------------------------------------------
            # Total loss
            # -----------------------------------------------------

            loss = (
                policy_loss
                + self.value_coef
                * value_loss
                - self.entropy_coef
                * entropy_loss
            )

            # -----------------------------------------------------
            # Gradient update
            # -----------------------------------------------------

            self.optimizer.zero_grad()

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                self.policy.parameters(),
                max_norm=0.5,
            )

            self.optimizer.step()

            total_policy_loss += (
                policy_loss.item()
            )

            total_value_loss += (
                value_loss.item()
            )

            total_entropy += (
                entropy_loss.item()
            )

        return {
            "policy_loss": (
                total_policy_loss
                / self.update_epochs
            ),
            "value_loss": (
                total_value_loss
                / self.update_epochs
            ),
            "entropy": (
                total_entropy
                / self.update_epochs
            ),
        }

    # =============================================================
    # TRAINING
    # =============================================================

    def train(self, n_episodes=100):
        """
        Train the PPO agent.

        Parameters
        ----------
        n_episodes : int, default=100
            Number of training episodes.

        Returns
        -------
        history : list[dict]
            Training statistics for each episode.
        """

        history = []

        progress_bar = tqdm(
            range(n_episodes),
            desc="PPO Training",
            unit="episode",
        )

        for episode in progress_bar:

            trajectory = self.collect_episode()

            update_info = self.update(
                trajectory
            )

            episode_reward = sum(
                trajectory["rewards"]
            )

            # ---------------------------------------------------------
            # Episode statistics
            # ---------------------------------------------------------

            final_accuracy = (
                self.env.current_accuracy
            )

            base_accuracy = (
                self.env.base_accuracy
            )

            improvement = (
                final_accuracy
                - base_accuracy
            )

            history.append({
                "episode": episode + 1,
                "reward": episode_reward,
                "base_accuracy": base_accuracy,
                "final_accuracy": final_accuracy,
                "improvement": improvement,
                "steps": len(
                    trajectory["rewards"]
                ),
                **update_info,
            })

            # ---------------------------------------------------------
            # Update progress bar
            # ---------------------------------------------------------

            progress_bar.set_postfix(
                reward=f"{episode_reward:.3f}",
                accuracy=f"{final_accuracy:.3f}",
                improvement=f"{improvement:+.3f}",
            )

        return history

    # =============================================================
    # EVALUATION
    # =============================================================

    def evaluate(self):
        """
        Evaluate the learned policy deterministically.

        Returns
        -------
        dict
            Final prompt, accuracy and selected actions.
        """

        state, info = (
            self.env.reset()
        )

        selected_actions = []

        done = False

        while not done:

            action_mask = (
                self.env.get_action_mask()
            )

            (
                action,
                _,
                _,
            ) = self.select_action(
                state,
                action_mask=action_mask,
                deterministic=True,
            )

            (
                next_state,
                reward,
                terminated,
                truncated,
                info,
            ) = self.env.step(action)

            selected_actions.append(
                action
            )

            done = (
                terminated
                or truncated
            )

            state = next_state

        return {
            "prompt": info["prompt"],
            "accuracy": info["accuracy"],
            "actions": selected_actions,
        }

In [41]:
llm = FrozenLLM(max_new_tokens=256,
                temperature=0.0)


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [42]:
from datasets import load_dataset

dataset = load_dataset(
    "openai/gsm8k",
    "main"
)

train_dataset = dataset["train"].select(
    range(50)
)

llm = FrozenLLM(max_new_tokens=256,
                temperature=0.0)

evaluator = GSM8KEvaluator(
    llm,
    train_dataset
)

env = PromptOptimizationEnv(
    base_prompt=BASE_PROMPT,
    evaluator=evaluator,
    max_steps=5
)

agent = PPO(
    env=env,
    learning_rate=1e-4,
    update_epochs=4
)



README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [43]:
history = agent.train(
    n_episodes=40
)

PPO Training:   0%|          | 0/40 [00:00<?, ?episode/s]

In [44]:
result = agent.evaluate()
print("\nFinal prompt:")
print(result['prompt'])

print("\nAccuracy:")
print(result["accuracy"])

print("\nSelected actions:")
print(result['actions'])


Final prompt:
Solve the following math problem.
Provide the final answer clearly.

Verify your answer before giving the final answer.

Explain your reasoning clearly.

Double-check your final answer.

Clearly state the final answer at the end.

Check your calculations carefully.

Accuracy:
0.22

Selected actions:
[2, 1, 6, 7, 3]


In [45]:
base_accuracy = evaluator.evaluate(
    BASE_PROMPT
)
print(f"Base accuracy: {base_accuracy:.4f}")

Base accuracy: 0.3800


In [46]:
result = agent.evaluate()
print(
    f"RL accuracy: {result["accuracy"]:.4f}"
)

RL accuracy: 0.2200


In [47]:
diff = result["accuracy"]-base_accuracy
print(diff)

-0.16


In [48]:
trajectory = agent.collect_episode()

print("Actions:", trajectory["actions"])
print("Rewards:", trajectory["rewards"])
print("Value:", trajectory["values"])

print(
    "Total reward: ",
    sum(trajectory["rewards"])
)


Actions: [1, 6, 5, 3, 2]
Rewards: [-0.16, 0.0, 0.0, -0.06, -0.17]
Value: [-0.13873669505119324, -0.18569937348365784, -0.19545608758926392, -0.2344890534877777, -0.27677813172340393]
Total reward:  -0.39


In [51]:
import json

chemin_fichier = "/content/output.json"

results = {
    "configuration": {
        "dataset_size": 50,
        "model": "Qwen/Qwen2.5-3B-Instruct",
        "learning_rate": 1e-4,
        "gamma": 0.99,
        "gae_lambda": 0.95,
        "clip_epsilon": 0.2,
        "value_coef": 0.5,
        "entropy_coef": 0.01,
        "update_epochs": 4,
        "hidden_dim": 128,
    },
    "PPO": {
        "accuracy": result["accuracy"],
        "actions": trajectory["actions"],
        "reward": trajectory["rewards"],
        "value": trajectory["values"]
    }
}

with open(chemin_fichier, "w", encoding="utf-8") as file:
    json.dump(results, file, indent=4, ensure_ascii=False)

print(f"\nResults saved to {chemin_fichier}")


Results saved to /content/output.json
